In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
fe = FeatureEngineeringClient(model_registry_uri="databricks-uc")

catalog = "main"
schema = "dbdemos_fs_travel"
table_name = "`{catalog}`.`{schema}`.travel_purchase"
model_name = "purchase_travel_model"
model_full_name = f"{catalog}.{schema}.{model_name}"
model_alias = "production"

batch_scoring = spark.table(table_name).select('user_id', 'destination_id', 'purchased', 'ts') \
    .where("ts >= '2022-11-01'")
scored_df = fe.score_batch(model_uri=f"models:/{model_full_name}@{model_alias}", df=batch_scoring, result_type="boolean")
display(scored_df.limit(100))

In [0]:
import mlflow.deployments

# Initialize MLflow deployment client for Databricks
client = mlflow.deployments.get_deploy_client("databricks")

# Get your actual model serving endpoint name from the previous step
ms_endpoint_name = f"{catalog}_{schema}_ms-travel-recommendation"[:50]

# Prepare sample inputs (these correspond to your lookup keys)
# You can query multiple user_id + destination_id pairs
response = client.predict(
    endpoint=ms_endpoint_name,
    inputs={
        "dataframe_records": [
            {"user_id": 1234, "destination_id": 12},
        ]
    },
)

print("Feature Serving Endpoint Response:")
print(response)